# 2.1 — What is RAG?

**RAG (Retrieval-Augmented Generation)** solves a key problem with LLMs:

| Problem | Solution |
|---------|----------|
| LLMs have a knowledge cutoff | Inject up-to-date documents at query time |
| LLMs hallucinate facts | Ground answers in real source documents |
| LLMs don't know your private data | Feed your own files to the model |

## How RAG Works

```
Your Documents
     ↓
  Chunking          (split into smaller pieces)
     ↓
  Embedding         (convert text → numbers / vectors)
     ↓
  Vector Store      (save vectors in a searchable database)
     ↓
  User Query  →  Retrieve relevant chunks
                      ↓
               LLM generates answer using those chunks
```

In [2]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## The Problem: LLM Without RAG

Ask the LLM about something it cannot know — private or recent information.

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.1', temperature=0)

# Ask about something the model doesn't know
response = llm.invoke('What is the refund policy for Acme Corp?')
print('Without RAG:')
print(response.content)

Without RAG:
I don't have specific information on the refund policy for Acme Corp. Companies often have their own policies, and without more context, it's difficult to provide a detailed answer. If you're looking for the refund policy of a specific Acme Corp, I recommend checking their official website or contacting their customer service directly. They should be able to provide you with the most accurate and up-to-date information.


## The Solution: LLM With RAG

We provide the relevant document — the model now has the context it needs.

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

# Simulate a retrieved document chunk
retrieved_context = """
Acme Corp Refund Policy (Updated 2025):
- Customers may return any product within 30 days for a full refund.
- Items must be unused and in original packaging.
- Digital products are non-refundable once downloaded.
- To initiate a return, contact support@acmecorp.com.
"""

messages = [
    SystemMessage(content=f'Answer using only this context:\n{retrieved_context}'),
    HumanMessage(content='What is the refund policy for Acme Corp?')
]

response = llm.invoke(messages)
print('With RAG:')
print(response.content)

With RAG:
According to the Acme Corp Refund Policy (Updated 2025), customers can return any product within 30 days for a full refund, as long as the items are unused and in their original packaging.


## Key Takeaway

RAG = **Retrieval** (find the right document chunks) + **Augmented Generation** (generate an answer using those chunks)

The LLM itself doesn't change — we just give it better context.

| Component | Tool Used |
|-----------|----------|
| Document Loader | `PyPDFLoader`, `TextLoader` |
| Text Splitter | `RecursiveCharacterTextSplitter` |
| Embeddings | `OllamaEmbeddings` |
| Vector Store | `Chroma` |
| LLM | `ChatOllama` |